# Σύστημα Β – Βήμα 3: τελική αξιολόγηση στο test
Εδώ χρησιμοποιούμε το test set **για πρώτη και μοναδική φορά**. Όλες οι επιλογές (είσοδοι, δίκτυο, κατώφλι) έχουν γίνει ήδη στο validation.

1. Φόρτωση του αποθηκευμένου δικτύου και εκπαίδευση των μοντέλων σύγκρισης (ίδια με το B02).
2. Κατώφλι απόφασης: επιλογή στο validation.
3. Όλα τα μοντέλα στο test.
4. Πίνακας σύγχυσης και καμπύλες ROC.
5. Δίκτυο και κανόνας στο test: πόσο συμφωνούν;
6. Ο κανόνας του δέντρου σε **πραγματικές μονάδες** (για το κείμενο της διπλωματικής).

Χρόνος: 1–2 λεπτά.

In [ ]:
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import keras
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             roc_curve, confusion_matrix)

DATA, MODELS, FIG = "../data/", "../models/", "../figures/"
SEED = 42

d = np.load(DATA + "intrusion_prepared.npz")
with open(MODELS + "intrusion_meta.json", encoding="utf-8") as f:
    meta = json.load(f)
assert "browser_type" in meta["cat_cols"], "Τα δεδομένα είναι από το παλιό B01 (8 είσοδοι) – τρέξε πρώτα το νέο B01"
X = {s: d[f"X_{s}"] for s in ("train", "val", "test")}
y = {s: d[f"y_{s}"].astype(int) for s in ("train", "val", "test")}
feature_names = meta["feature_names"]
short = [f.split("__")[-1] for f in feature_names]
preprocessor = joblib.load(MODELS + "intrusion_preprocessor.joblib")

model = keras.models.load_model(MODELS + "intrusion_mlp.keras", compile=False)
p_nn = {s: model.predict(X[s], batch_size=1024, verbose=0).ravel() for s in ("val", "test")}
print("Σχήματα:", {s: X[s].shape for s in X})
print("Δίκτυο:", json.load(open(MODELS + "intrusion_mlp_config.json", encoding="utf-8")))

## Β3.1 Μοντέλα σύγκρισης
Εκπαιδεύονται ξανά στο train με τις ίδιες ρυθμίσεις και τον ίδιο σπόρο με το B02, οπότε βγαίνουν τα ίδια.
Έλεγχος: στο validation πρέπει να ξαναβγούν τα νούμερα του B02 (π.χ. δέντρο βάθους 5: accuracy 0.8847, Gradient Boosting: 0.8889, νευρωνικό: 0.8847).

In [ ]:
RULE, TREE5, NN = "Κανόνας 3 ερωτήσεων (δέντρο βάθους 3)", "Δέντρο βάθους 5", "Νευρωνικό δίκτυο"
models = {RULE: DecisionTreeClassifier(max_depth=3, random_state=SEED),
          TREE5: DecisionTreeClassifier(max_depth=5, random_state=SEED),
          "Logistic Regression": LogisticRegression(max_iter=1000),
          "Random Forest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED),
          "Gradient Boosting": GradientBoostingClassifier(random_state=SEED)}
prob = {}
for name, m in models.items():
    m.fit(X["train"], y["train"])
    prob[name] = {s: m.predict_proba(X[s])[:, 1] for s in ("val", "test")}
prob[NN] = p_nn

def metrics(y_true, p, t=0.5):
    pred = p >= t
    return {"accuracy": accuracy_score(y_true, pred), "precision": precision_score(y_true, pred, zero_division=0),
            "recall (επιθέσεις)": recall_score(y_true, pred), "F1": f1_score(y_true, pred),
            "FPR (ψευδείς συναγερμοί)": ((pred == 1) & (y_true == 0)).sum() / (y_true == 0).sum(),
            "ROC-AUC": roc_auc_score(y_true, p)}

print("Έλεγχος στο validation:")
print(pd.DataFrame({name: metrics(y["val"], p["val"]) for name, p in prob.items()}).T[["accuracy", "recall (επιθέσεις)", "FPR (ψευδείς συναγερμοί)"]].round(4).to_string())

## Β3.2 Κατώφλι απόφασης
Όπως στο Σύστημα Α: το κατώφλι του δικτύου διαλέγεται στο **validation** (μέγιστη ακρίβεια). Αν το κέρδος έναντι του 0.5 είναι αμελητέο, κρατάμε το 0.5.

In [ ]:
ths = np.round(np.arange(0.05, 1.0, 0.05), 2)
acc_val = pd.Series([accuracy_score(y["val"], p_nn["val"] >= t) for t in ths], index=ths)
T_VAL = float(acc_val.idxmax())
gain = acc_val.max() - acc_val.loc[0.5]
print(f"Κατώφλι με μέγιστη ακρίβεια στο validation: {T_VAL} | κέρδος έναντι του 0.5: {gain:+.4f}")
THRESHOLD = T_VAL if gain >= 0.005 else 0.5
print(f"Κατώφλι που κρατάμε: {THRESHOLD}" + ("  (το κέρδος είναι αμελητέο)" if THRESHOLD == 0.5 and T_VAL != 0.5 else ""))

## Β3.3 Όλα τα μοντέλα στο test
Αυτά είναι τα νούμερα που μπαίνουν στη διπλωματική. Για το νευρωνικό χρησιμοποιείται το κατώφλι της ενότητας Β3.2· για τα υπόλοιπα το 0.5.

In [ ]:
final = {name: metrics(y["test"], p["test"], THRESHOLD if name == NN else 0.5) for name, p in prob.items()}
final_val = {name: metrics(y["val"], p["val"], THRESHOLD if name == NN else 0.5) for name, p in prob.items()}
final_table = pd.DataFrame(final).T.round(4)
final_table.to_csv(MODELS + "intrusion_final_test.csv")
print("TEST"); print(final_table.to_string())

cmp_vt = pd.DataFrame({"accuracy validation": {n: final_val[n]["accuracy"] for n in final},
                       "accuracy test": {n: final[n]["accuracy"] for n in final}})
cmp_vt["διαφορά"] = cmp_vt["accuracy test"] - cmp_vt["accuracy validation"]
print("\nValidation έναντι test (αν η διαφορά είναι μικρή, το validation ήταν αξιόπιστος οδηγός):")
print(cmp_vt.round(4).to_string())

## Β3.4 Πίνακας σύγχυσης του δικτύου και καμπύλες ROC (test)

In [ ]:
pred_nn = (p_nn["test"] >= THRESHOLD).astype(int)
cm = confusion_matrix(y["test"], pred_nn, labels=[0, 1])
cmn = cm / cm.sum(1, keepdims=True)
names = ["Κανονική", "Επίθεση"]
annot = np.array([[f"{cmn[i, j]:.1%}\n{cm[i, j]}" for j in range(2)] for i in range(2)])
fig, ax = plt.subplots(figsize=(4.8, 4))
sns.heatmap(cmn, annot=annot, fmt="", cmap="Blues", vmin=0, vmax=1, cbar=False,
            xticklabels=names, yticklabels=names, annot_kws={"size": 11}, ax=ax)
ax.set_xlabel("Πρόβλεψη"); ax.set_ylabel("Πραγματικό"); ax.set_title("Σύστημα Β – νευρωνικό δίκτυο (test)")
plt.yticks(rotation=0); plt.tight_layout(); plt.savefig(FIG + "B04_intrusion_confusion_matrix.png", dpi=200); plt.show()

styles = {NN: ("#eb6834", "-"), "Gradient Boosting": ("#2a78d6", "-"), TREE5: ("#1baf7a", "--"),
          RULE: ("#8a8a86", ":"), "Logistic Regression": ("#b9b8b4", "-.")}
fig, ax = plt.subplots(figsize=(6.2, 5))
for name, (color, ls) in styles.items():
    fpr, tpr, _ = roc_curve(y["test"], prob[name]["test"])
    ax.plot(fpr, tpr, color=color, ls=ls, lw=2, label=f"{name} (AUC = {final[name]['ROC-AUC']:.3f})")
ax.plot([0, 1], [0, 1], color="gray", lw=1, ls=(0, (1, 3)), label="Τυχαία πρόβλεψη")
ax.set_xlabel("Ψευδείς συναγερμοί (FPR)"); ax.set_ylabel("Επιθέσεις που εντοπίζονται (recall)")
ax.set_title("Σύστημα Β – καμπύλες ROC (test)")
ax.grid(alpha=0.3); ax.spines[["top", "right"]].set_visible(False); ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.savefig(FIG + "B05_intrusion_roc_test.png", dpi=200); plt.show()

## Β3.5 Δίκτυο, κανόνας και δέντρο στο test: πόσο συμφωνούν;

In [ ]:
pred = {name: (prob[name]["test"] >= (THRESHOLD if name == NN else 0.5)).astype(int) for name in (NN, RULE, TREE5)}
yt = y["test"]
for other in (RULE, TREE5):
    diff = pred[NN] != pred[other]
    print(f"Δίκτυο και «{other}»: ίδια απόφαση στο {1 - diff.mean():.1%} των ροών | διαφέρουν σε {int(diff.sum())} ροές, "
          f"εκ των οποίων σωστό το δίκτυο σε {int((diff & (pred[NN] == yt)).sum())}")

hidden = (pred[RULE] == 0) & (yt == 1)
print(f"\nΕπιθέσεις που ο κανόνας των 3 ερωτήσεων δεν πιάνει: {int(hidden.sum())} | από αυτές βρίσκει το δίκτυο: {int((hidden & (pred[NN] == 1)).sum())}")
missed = (pred[NN] == 0) & (yt == 1)
print(f"Επιθέσεις που δεν βρίσκει το δίκτυο: {int(missed.sum())} από {int(yt.sum())} ({missed.sum() / yt.sum():.1%})")

## Β3.6 Ο κανόνας σε πραγματικές μονάδες
Τα δέντρα του B01 έδειχναν τα όρια στην κλίμακα μετά τον μετασχηματισμό (π.χ. `failed_logins > 0.95`). Εδώ τα μετατρέπουμε πίσω στις αρχικές μονάδες,
με τον ίδιο preprocessor, ώστε ο κανόνας να γράφεται στη διπλωματική όπως θα τον διάβαζε ένας διαχειριστής δικτύου.
Σε κάθε φύλλο φαίνεται πόσες ροές του train καταλήγουν εκεί και τι ποσοστό τους είναι επιθέσεις.

In [ ]:
def to_original(fname, t):
    prefix, col = fname.split("__", 1)
    if prefix == "num":
        sc = preprocessor.named_transformers_["num"]
        i = list(sc.feature_names_in_).index(col)
        return col, t * sc.scale_[i] + sc.mean_[i]
    if prefix == "log":
        sc = preprocessor.named_transformers_["log"].named_steps["scale"]
        i = list(sc.feature_names_in_).index(col)
        return col, float(np.expm1(t * sc.scale_[i] + sc.mean_[i]))
    if prefix == "bin":
        return col, t
    for c in meta["cat_cols"]:                                  # one-hot: «στήλη_τιμή»
        if col.startswith(c + "_"):
            return (c, col[len(c) + 1:]), None
    return col, t

def print_rules(tree, depth=0, node=0):
    t = tree.tree_
    pad = "    " * depth
    if t.children_left[node] == -1:
        n, share = int(t.n_node_samples[node]), t.value[node][0][1] / t.value[node][0].sum()
        print(f"{pad}→ {'ΕΠΙΘΕΣΗ' if share >= 0.5 else 'κανονική'}   ({n} ροές του train, {share:.0%} επιθέσεις)")
        return
    col, thr = to_original(feature_names[t.feature[node]], t.threshold[node])
    if thr is None:
        cond_l, cond_r = f"{col[0]} ≠ {col[1]}", f"{col[0]} = {col[1]}"
    else:
        cond_l, cond_r = f"{col} ≤ {thr:.3g}", f"{col} > {thr:.3g}"
    print(f"{pad}αν {cond_l}:"); print_rules(tree, depth + 1, t.children_left[node])
    print(f"{pad}αν {cond_r}:"); print_rules(tree, depth + 1, t.children_right[node])

for name in (RULE, TREE5):
    print(f"===== {name} =====")
    print_rules(models[name]); print()